In [ ]:
import datetime
import os

import pandas as pd

from modules.predictor.data.utils import prepare_data_for_regressors, custom_data_split, combine_split
from modules.predictor.training_and_evaluation.train_eval_pipeline import batch_train_and_eval

In [ ]:
num_splits = 4
num_bins = 4
random_state = 42

features_all = ['flatness', 'PS', '%C', '%N', '%O', 'MolLogP', 'num_atoms', 'num_bonds', 'num_aromatic_rings',
                'num_heteroatoms', 'num_rotatable_bonds', 'num_h_acceptors', 'num_h_donors', 'tpsa', 'mol_wt',
                'symmetry_C2', 'symmetry_C2h', 'symmetry_C2v', 'symmetry_Cs', 'symmetry_D2', 'symmetry_D2h']
target = 'capacity_max'

# Model training and eval - only experts1 data

## Data preparation and split

In [ ]:
df_experts1 = pd.read_csv('../../../data/processed_selected_custom_features/data_experts1.csv')

df_experts1.head()

In [ ]:
folds_experts1 = custom_data_split(df_experts1, target, num_splits, num_bins, random_state)

for (train, test) in folds_experts1:
    print(len(train), len(test))

In [ ]:
df_experts1.drop(columns=['smiles'], inplace=True)

cat_features = ['symmetry']
num_features = [f for f in df_experts1.columns if f not in [target]]

df_experts1 = prepare_data_for_regressors(df_experts1, num_features, cat_features)
features = [f for f in df_experts1 if f in features_all]
df_experts1.head()

## Model evaluation

In [ ]:
date = datetime.date.today().strftime('%d-%m-%Y')
save_dir = f'../../../results/expert1/{date}/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
batch_train_and_eval(df_experts1, 'expert1', folds_experts1, target, features, 'grid_search', save_dir)

# Model training and eval - only expert data (experts1 and experts2)

## Data preparation and split

In [ ]:
df_experts1 = pd.read_csv('../../../data/processed_selected_custom_features/data_experts1.csv')
df_experts2 = pd.read_csv('../../../data/processed_selected_custom_features/data_experts2.csv')

In [ ]:
folds_experts1 = custom_data_split(df_experts1, target, num_splits, num_bins, random_state)
folds_experts2 = custom_data_split(df_experts2, target, num_splits, num_bins, random_state)

folds_experts, df_experts = combine_split(df_experts1, folds_experts1, df_experts2, folds_experts2)

print(len(df_experts))
for (train, test) in folds_experts:
    print(len(train), len(test))

In [ ]:
df_experts.drop(columns=['smiles'], inplace=True)

cat_features = ['symmetry']
num_features = [f for f in df_experts.columns if f not in [target]]

df_experts = prepare_data_for_regressors(df_experts, num_features, cat_features)
features = [f for f in df_experts if f in features_all]
df_experts.head()

## Model evaluation

In [ ]:
date = datetime.date.today().strftime('%d-%m-%Y')
save_dir = f'../../../results/expert/{date}/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
batch_train_and_eval(df_experts, 'expert', folds_experts, target, features, 'grid_search', save_dir)

# Model training and eval - all datasets (experts1, experts2, zhu, saad)

## Data preparation and split

In [ ]:
df_experts1 = pd.read_csv('../../../data/processed_selected_custom_features/data_experts1.csv')
df_experts2 = pd.read_csv('../../../data/processed_selected_custom_features/data_experts2.csv')
df_zhu = pd.read_csv('../../../data/processed_selected_custom_features/data_zhu.csv')
df_saad = pd.read_csv('../../../data/processed_selected_custom_features/data_saad.csv')

df_rest = pd.concat([df_zhu, df_saad, df_experts2], ignore_index=True)
print(len(df_rest))

In [ ]:
folds_experts1 = custom_data_split(df_experts1, target, num_splits, num_bins, random_state)
folds_rest = custom_data_split(df_rest, target, num_splits, num_bins, random_state)

folds_all, df_all = combine_split(df_experts1, folds_experts1, df_rest, folds_rest)

print(len(df_all))
for (train, test) in folds_all:
    print(len(train), len(test))

In [ ]:
df_all.drop(columns=['smiles'], inplace=True)

cat_features = ['symmetry']
num_features = [f for f in df_all.columns if f not in [target]]

df_all = prepare_data_for_regressors(df_all, num_features, cat_features)
features = [f for f in df_experts if f in features_all]
df_all.head()

## Model evaluation

In [ ]:
date = datetime.date.today().strftime('%d-%m-%Y')
save_dir = f'../../../results/all/{date}/'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
batch_train_and_eval(df_all, 'all', folds_all, target, features, 'grid_search', save_dir)